<a href="https://colab.research.google.com/github/kroschenko/multi_domain_model/blob/main/Uncertainty_Gated_Multimodal_Fusion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Heteroscedastic Uncertainty as a Dynamic Gate for Robust Multimodal Emotion Recognition

In [ ]:
# ## 📦 0. Setup & Auth (Fixed)
import os, sys, gc, warnings, shutil, time
from pathlib import Path
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics import f1_score, accuracy_score
from sklearn.utils.class_weight import compute_class_weight
from dataclasses import dataclass  # ← ВАЖНО: был пропущен импорт!
import matplotlib.pyplot as plt, seaborn as sns
from tqdm import tqdm
warnings.filterwarnings('ignore')

# === GOOGLE DRIVE MOUNT (с обработкой ошибки) ===
from google.colab import drive

try:
    print("🔐 Connecting to Google Drive...")
    print("👉 Если появится ссылка — откройте её, авторизуйтесь и вставьте код ниже:")
    drive.mount('/content/drive', force_remount=True)  # force_remount=True решает 90% проблем
    DRIVE_ROOT = Path("/content/drive/MyDrive")

    # ✅ Проверяем, что монтирование прошло успешно
    if (DRIVE_ROOT / "MyDrive").exists() or (DRIVE_ROOT / "erc_project").exists():
        print("✅ Drive mounted successfully")
    else:
        raise FileNotFoundError("Drive root not found")

except Exception as e:
    print(f"⚠️ Drive mount failed: {e}")
    print("🔄 Fallback: using local Colab storage")
    DRIVE_ROOT = Path("/content")  # fallback на локальное хранилище

# 🔧 Папка для кэша (на Drive или локально)
EXP1_CACHE = DRIVE_ROOT / "exp1_article_cache"
EXP1_CACHE.mkdir(exist_ok=True)
print(f"💾 Cache dir: {EXP1_CACHE}")

# === DEVICE & SEED ===
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.set_default_dtype(torch.float32)

def set_seed(s: int):
    torch.manual_seed(s)
    np.random.seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed(42)
print(f"🔬 Device: {device} | RAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB" if device.type=='cuda' else f"🔬 Device: {device}")

🔐 Connecting to Google Drive...
👉 Если появится ссылка — откройте её, авторизуйтесь и вставьте код ниже:
Mounted at /content/drive
✅ Drive mounted successfully
💾 Cache dir: /content/drive/MyDrive/exp1_article_cache
🔬 Device: cuda | RAM: 15.6 GB


In [ ]:
###  1. Configuration (Article #1 Focus)
# Изолированный конфиг только для Uncertainty-Gated Fusion эксперимента.

@dataclass
class Exp1Config:
    # Data
    max_utterances: int = 5
    hidden_dim: int = 256
    dropout: float = 0.2
    num_speakers: int = 20

    # Modalities & Fusion
    use_audio: bool = True
    use_video: bool = True
    fusion_type: str = "uncertainty_gate"  # fixed for Exp1
    uncertainty_loss_weight: float = 0.5

    # Training
    batch_size: int = 16
    epochs: int = 5
    lr: float = 1e-3
    weight_decay: float = 0.01
    early_stopping_patience: int = 3
    temperature_scaling: bool = True  # post-hoc calibration

    # Experiment
    n_seeds: int = 3
    experiment_name: str = "exp1_uncertainty_gate"
    cache_dir: str = "./exp1_cache"
    text_model: str = "microsoft/deberta-v3-base"
    freeze_text: bool = True

config = Exp1Config()
Path(config.cache_dir).mkdir(exist_ok=True)
print(f"✅ Config loaded: {config.experiment_name} | Fusion: {config.fusion_type}")

✅ Config loaded: exp1_uncertainty_gate | Fusion: uncertainty_gate


In [ ]:
# ## 2. Feature Extraction + Validation
# Извлекает аудио/видео, сохраняет в EXP1_CACHE, сразу считает статистику.

import librosa, cv2, tempfile, subprocess
from transformers import AutoProcessor, Wav2Vec2Model
from torchvision import models, transforms

class FeatureExtractor:
    def __init__(self):
        print("🎵 Loading Wav2Vec2...")
        self.wav_proc = AutoProcessor.from_pretrained("facebook/wav2vec2-base-960h")
        self.wav_model = Wav2Vec2Model.from_pretrained("facebook/wav2vec2-base-960h").eval().to(device)
        for p in self.wav_model.parameters(): p.requires_grad = False

        print("🎥 Loading ResNet-18...")
        self.resnet = models.resnet18(weights='IMAGENET1K_V1')
        self.resnet.fc = nn.Identity()
        self.resnet.eval().to(device)
        for p in self.resnet.parameters(): p.requires_grad = False

        self.img_tf = transforms.Compose([
            transforms.ToPILImage(), transforms.Resize((224,224)),
            transforms.ToTensor(), transforms.Normalize([0.485,0.456,0.406], [0.229,0.224,0.225])
        ])

    def extract_audio(self, video_path, uid):
        cache_file = EXP1_CACHE / f"{uid}_audio.npy"
        if cache_file.exists(): return np.load(cache_file)
        try:
            with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as tmp:
                cmd = ['ffmpeg', '-i', video_path, '-vn', '-acodec', 'pcm_s16le', '-ar', '16000', '-ac', '1', tmp.name, '-y', '-loglevel', 'error']
                subprocess.run(cmd, check=True, timeout=30, capture_output=True)
                y, sr = librosa.load(tmp.name, sr=16000, duration=4.0)
            inputs = self.wav_proc(y, sampling_rate=16000, return_tensors="pt").to(device)
            with torch.no_grad():
                out = self.wav_model(**inputs).last_hidden_state
                feat = torch.cat([out[:,0], out.mean(1), out.std(1)], dim=-1).squeeze().cpu().numpy()
            np.save(cache_file, feat)
            return feat
        except Exception as e:
            np.save(cache_file, np.zeros(2304, dtype=np.float32))
            return np.zeros(2304)

    def extract_video(self, video_path, uid):
        cache_file = EXP1_CACHE / f"{uid}_video.npy"
        if cache_file.exists(): return np.load(cache_file)
        try:
            cap = cv2.VideoCapture(video_path)
            total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
            idxs = np.linspace(0, max(total-1,0), 8, dtype=int)
            feats = []
            for i in idxs:
                cap.set(cv2.CAP_PROP_POS_FRAMES, i)
                ret, frame = cap.read()
                if ret:
                    img = self.img_tf(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)).unsqueeze(0).to(device)
                    with torch.no_grad(): feats.append(self.resnet(img).squeeze().cpu().numpy())
            cap.release()
            res = np.mean(feats, axis=0) if feats else np.zeros(512)
            np.save(cache_file, res)
            return res
        except:
            np.save(cache_file, np.zeros(512))
            return np.zeros(512)

extractor = FeatureExtractor()
print("✅ Extractor ready")

🎵 Loading Wav2Vec2...


preprocessor_config.json:   0%|          | 0.00/159 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/163 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

vocab.json:   0%|          | 0.00/291 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/210 [00:00<?, ?it/s]

Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base-960h
Key               | Status     | 
------------------+------------+-
lm_head.bias      | UNEXPECTED | 
lm_head.weight    | UNEXPECTED | 
masked_spec_embed | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


🎥 Loading ResNet-18...
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 154MB/s]

✅ Extractor ready


In [ ]:
# ## 🗃️ 3. Data Loading (MELD via Kaggle) + Dataset + Diagnostics

# === ЗАГРУЗКА ДАННЫХ (ВАШ КОД) ===
try:
    import kagglehub
    print("📦 Downloading MELD from Kaggle...")
    path = kagglehub.dataset_download("bhandariprakanda/meld-emotion-recognition")
    video_base = os.path.join(path, "MELD.Raw", "MELD.Raw")
    csv_path = os.path.join(path, "JSON files", "JSON files", "Updated CSV")

    train_df = pd.read_csv(os.path.join(csv_path, "train_sent_emo_cleaned.csv"))
    test_df = pd.read_csv(os.path.join(csv_path, "test_sent_emo_cleaned.csv"))

    # Создаём video_path колонки
    train_df["video_path"] = train_df.apply(
        lambda r: os.path.join(video_base, "train", "train_splits",
                              f"dia{int(r['Dialogue_ID'])}_utt{int(r['Utterance_ID'])}.mp4"), axis=1)
    test_df["video_path"] = test_df.apply(
        lambda r: os.path.join(video_base, "test", "output_repeated_splits_test",
                              f"dia{int(r['Dialogue_ID'])}_utt{int(r['Utterance_ID'])}.mp4"), axis=1)

    # Валидационный сплит
    val_df = train_df.sample(frac=0.15, random_state=42)
    train_df = train_df.drop(val_df.index).reset_index(drop=True)
    val_df = val_df.reset_index(drop=True)

    print(f"📊 Loaded MELD: Train={len(train_df)}, Val={len(val_df)}, Test={len(test_df)}")

except Exception as e:
    print(f"⚠️ Using dummy data (error: {e})")
    train_df = pd.DataFrame({
        "Dialogue_ID": [1]*50 + [2]*50,
        "Utterance_ID": list(range(50))*2,
        "Utterance": ["Hello world"]*100,
        "Speaker": ["S1", "S2"]*50,
        "Emotion": ["neutral", "joy"]*50,
        "video_path": [None]*100
    })
    test_df = train_df.iloc[:20].copy()
    val_df = train_df.iloc[20:40].copy()

# Фильтруем только 7 основных эмоций
EMOTIONS = ["neutral","joy","sadness","anger","surprise","disgust","fear"]
train_df = train_df[train_df["Emotion"].isin(EMOTIONS)].reset_index(drop=True)
val_df = val_df[val_df["Emotion"].isin(EMOTIONS)].reset_index(drop=True)

# === DATASET (использует video_path из df) ===
class DiagDataset(Dataset):
    def __init__(self, df, extractor, cache_dir: Path):
        self.df = df.reset_index(drop=True)
        self.extractor = extractor
        self.cache_dir = cache_dir
        self.emotions = sorted(df["Emotion"].unique())
        self.emo2idx = {e:i for i,e in enumerate(self.emotions)}

        print(f"🔍 Pre-extracting features for {len(self.df)} samples...")
        for _, row in tqdm(self.df.iterrows(), total=len(self.df), leave=False):
            uid = f"dia{int(row['Dialogue_ID'])}_utt{int(row['Utterance_ID'])}"
            v_path = row.get("video_path")

            if v_path and os.path.exists(v_path):
                self.extractor.extract_audio(v_path, uid)
                self.extractor.extract_video(v_path, uid)
            else:
                # Fallback: нулевые фичи
                for mod, dim in [("audio", 2304), ("video", 512)]:
                    np.save(cache_dir/f"{uid}_{mod}.npy", np.zeros(dim, dtype=np.float32))

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        uid = f"dia{int(row['Dialogue_ID'])}_utt{int(row['Utterance_ID'])}"
        return {
            "text": str(row["Utterance"]),
            "audio": torch.from_numpy(np.load(self.cache_dir/f"{uid}_audio.npy")).float(),
            "video": torch.from_numpy(np.load(self.cache_dir/f"{uid}_video.npy")).float(),
            "label": torch.tensor(self.emo2idx[row["Emotion"]]),  # ← ключ "label"
            "uid": uid
        }

def collate_diag(batch):
    return {
        "text": [b["text"] for b in batch],
        "audio": torch.stack([b["audio"] for b in batch]),
        "video": torch.stack([b["video"] for b in batch]),
        "label": torch.stack([b["label"] for b in batch]),  # ← ключ "label"
        "uid": [b["uid"] for b in batch]
    }

# Создаём датасеты
train_ds = DiagDataset(train_df, extractor, EXP1_CACHE)
val_ds = DiagDataset(val_df, extractor, EXP1_CACHE)
dl_train = DataLoader(train_ds, batch_size=config.batch_size, shuffle=True, collate_fn=collate_diag)
dl_val = DataLoader(val_ds, batch_size=config.batch_size, shuffle=False, collate_fn=collate_diag)

# 🔍 DIAGNOSTIC REPORT
print("\n" + "="*40 + " PRE-TRAIN DIAGNOSTICS " + "="*40)
train_labels = np.array([d["label"].item() for d in train_ds])
print(f"📊 Classes: {len(train_ds.emotions)} | Train: {len(train_ds)} | Val: {len(val_ds)}")
print(f"📈 Label Distribution:\n{pd.Series(train_labels).map({i:e for e,i in train_ds.emo2idx.items()}).value_counts().sort_index()}")

# Class weights
cw = compute_class_weight('balanced', classes=np.arange(len(train_ds.emotions)), y=train_labels)
print(f"⚖️ Class Weights: {cw}")

# Feature stats
a_batch = next(iter(dl_train))["audio"]
v_batch = next(iter(dl_train))["video"]
for name, feat in [("Audio", a_batch), ("Video", v_batch)]:
    print(f"\n🔎 {name} Features [{feat.shape[1]} dim]:")
    print(f"   Mean: {feat.mean():.3f} | Std: {feat.std():.3f} | Min: {feat.min():.3f} | Max: {feat.max():.3f}")
    print(f"   Zeros: {(feat.abs()<1e-6).float().mean()*100:.1f}% | NaNs: {torch.isnan(feat).any()}")
print("="*100)

📦 Downloading MELD from Kaggle...
Using Colab cache for faster access to the 'meld-emotion-recognition' dataset.
📊 Loaded MELD: Train=8490, Val=1498, Test=2610
🔍 Pre-extracting features for 8490 samples...


  7%|▋         | 554/8490 [13:29<2:27:06,  1.11s/it]

In [ ]:
def compute_ece(probs: np.ndarray, labels: np.ndarray, n_bins: int = 10) -> float:
    confidences = np.max(probs, axis=1)
    predictions = np.argmax(probs, axis=1)
    bin_boundaries = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        mask = (confidences >= bin_boundaries[i]) & (confidences < bin_boundaries[i+1])
        if mask.sum() > 0:
            acc = (predictions[mask] == labels[mask]).mean()
            conf = confidences[mask].mean()
            ece += np.abs(acc - conf) * mask.sum() / len(labels)
    return float(ece)

def compute_brier(probs: np.ndarray, labels: np.ndarray, n_classes: int) -> float:
    one_hot = np.eye(n_classes)[labels]
    return float(np.mean(np.sum((probs - one_hot)**2, axis=1)))

In [ ]:
# ## 🧠 4. Model Architecture: Uncertainty-Gated Fusion
# Ядро статьи №1. Исправлены: размерность аудио, нормализация, параллельный гейтинг.

class AdaptiveGatedFusion(nn.Module):
    """
    Uncertainty-aware gated attention.
    gate = σ(quality) × σ(-log_var), clamped to [0.15, 0.85] for stability.
    """
    def __init__(self, dim: int, n_heads: int = 4, dropout: float = 0.1):
        super().__init__()
        self.quality_proj = nn.Linear(dim, 1)
        self.attn = nn.MultiheadAttention(dim, n_heads, dropout=dropout, batch_first=True)
        self.norm = nn.LayerNorm(dim)
        self.dropout = nn.Dropout(dropout)
        self.gate_bias = nn.Parameter(torch.tensor(0.0))  # learnable shift

    def forward(self, text_q: torch.Tensor, mod_kv: torch.Tensor, log_var: torch.Tensor):
        # Quality score
        quality = torch.sigmoid(self.quality_proj(mod_kv) + self.gate_bias)  # [B, 1]

        # Cross-attention: text attends to modality
        attn_out, _ = self.attn(
            text_q.unsqueeze(1),           # query [B, 1, D]
            mod_kv.unsqueeze(1),           # key   [B, 1, D]
            mod_kv.unsqueeze(1),           # value [B, 1, D]
        )
        attended = self.norm(text_q + self.dropout(attn_out.squeeze(1)))  # [B, D]

        # Uncertainty weighting (stabilized)
        unc_weight = 1.0 / (1.0 + torch.exp(torch.clamp(log_var, min=-5, max=5)))
        unc_weight = torch.clamp(unc_weight, min=0.15, max=0.85)  # prevent gate collapse

        # Final gate & fusion
        gate = quality * unc_weight  # [B, 1]
        fused = gate * attended + (1 - gate) * text_q
        return fused, gate.squeeze(-1)  # [B, D], [B]


class UncertaintyGateModel(nn.Module):
    def __init__(self, config: Exp1Config, num_classes: int = 7):
        super().__init__()
        self.config = config

        # Text encoder (DeBERTa)
        self.text_tok = AutoTokenizer.from_pretrained(config.text_model)
        self.text_enc = AutoModel.from_pretrained(config.text_model)

        # ✅ FIX: Unfreeze last 2 layers for fine-tuning (frozen DeBERTa + random features = collapse)
        if config.freeze_text:
            for p in self.text_enc.parameters(): p.requires_grad = False
            for layer in self.text_enc.encoder.layer[-2:]:
                for p in layer.parameters(): p.requires_grad = True

        # Projections: note AUDIO_DIM = 2304 (CLS+mean+std), not 768!
        AUDIO_DIM = 2304  # ← КРИТИЧНО: новый размер аудио-фич
        VIDEO_DIM = 512
        TEXT_DIM = 768

        self.text_proj = nn.Linear(TEXT_DIM, config.hidden_dim)
        self.audio_proj = nn.Linear(AUDIO_DIM, config.hidden_dim) if config.use_audio else None
        self.video_proj = nn.Linear(VIDEO_DIM, config.hidden_dim) if config.use_video else None

        # Fusion modules
        self.fusion_a = AdaptiveGatedFusion(config.hidden_dim) if config.use_audio else None
        self.fusion_v = AdaptiveGatedFusion(config.hidden_dim) if config.use_video else None

        # Uncertainty head
        self.unc_head = nn.Sequential(
            nn.Linear(config.hidden_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

        # Classifier
        self.classifier = nn.Sequential(
            nn.Linear(config.hidden_dim, config.hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(config.dropout),
            nn.Linear(config.hidden_dim // 2, num_classes)
        )

        # Temperature scaling for calibration
        self.temperature = nn.Parameter(torch.ones(1) * 1.5) if config.temperature_scaling else None

    def forward(self, batch: Dict, training: bool = False):
        device = next(self.parameters()).device
        B = len(batch["text"])

        # === TEXT ===
        tok = self.text_tok(batch["text"], padding=True, truncation=True, max_length=64, return_tensors="pt")
        tok = {k: v.to(device) for k, v in tok.items()}
        with torch.set_grad_enabled(not self.config.freeze_text or training):
            text_cls = self.text_enc(**tok).last_hidden_state[:, 0]  # [B, 768]
        text_cls = self.text_proj(text_cls)  # [B, hidden_dim]

        # === MODALITIES (with on-the-fly normalization) ===
        # ✅ FIX: Normalize before projection to stabilize training
        if self.audio_proj is not None:
            a_raw = batch["audio"].to(device)
            a_norm = (a_raw - a_raw.mean(dim=1, keepdim=True)) / (a_raw.std(dim=1, keepdim=True) + 1e-8)
            a = self.audio_proj(a_norm)
        else:
            a = torch.zeros(B, self.config.hidden_dim, device=device)

        if self.video_proj is not None:
            v_raw = batch["video"].to(device)
            v_norm = (v_raw - v_raw.mean(dim=1, keepdim=True)) / (v_raw.std(dim=1, keepdim=True) + 1e-8)
            v = self.video_proj(v_norm)
        else:
            v = torch.zeros(B, self.config.hidden_dim, device=device)

        # === UNCERTAINTY ===
        log_var = torch.clamp(self.unc_head(text_cls), min=-5.0, max=2.0)  # [B, 1]

        # === PARALLEL GATED FUSION ✅ FIX: не последовательно, а параллельно ===
        fused = text_cls.clone()
        gates = {}

        if self.fusion_a is not None:
            fused_a, gate_a = self.fusion_a(fused, a, log_var)
            gates["audio"] = gate_a
            fused = fused_a  # update fused after audio

        if self.fusion_v is not None:
            # ✅ Важно: применяем видео-гейт к уже аудио-фузженному тексту
            fused, gate_v = self.fusion_v(fused, v, log_var)
            gates["video"] = gate_v

        # === CLASSIFIER ===
        logits = self.classifier(fused)

        # Temperature scaling (for calibration)
        if self.temperature is not None:
            logits = logits / self.temperature.clamp(min=0.1)

        return logits, log_var, gates

In [ ]:
# Вставьте эту ячейку после Cell 4, перед тренировкой:
model = UncertaintyGateModel(config, num_classes=len(train_ds.emotions)).to(device)
batch = next(iter(dl_train))
batch = {k: v.to(device) if isinstance(v, torch.Tensor) else v for k,v in batch.items()}
logits, log_var, gates = model(batch, training=False)
assert logits.shape[1] == len(train_ds.emotions), "Logits dim mismatch!"
assert "audio" in gates and "video" in gates, "Gates dict missing keys!"
print("✅ Model sanity check PASSED")

In [ ]:
# ## 🔍 5. Diagnostic Forward Pass (FIXED)
model = UncertaintyGateModel(config, num_classes=len(train_ds.emotions)).to(device)
model.eval()
batch = next(iter(dl_train))
batch = {k: v.to(device) if isinstance(v, torch.Tensor) else v for k,v in batch.items()}

with torch.no_grad():
    logits, log_var, gates = model(batch, training=False)  # ← gates это dict
    probs = torch.softmax(logits, 1)
    preds = logits.argmax(1)

print("\n" + "="*40 + " FORWARD DIAGNOSTICS " + "="*40)
print(f"🔢 Logits range: [{logits.min().item():.2f}, {logits.max().item():.2f}]")
print(f"📊 Predicted distribution: {torch.bincount(preds, minlength=7).tolist()}")
print(f"🎯 True distribution:      {torch.bincount(batch['label'], minlength=7).tolist()}")  # ← batch['label']
print(f"🚪 Gate A: mean={gates['audio'].mean().item():.3f} | Gate V: mean={gates['video'].mean().item():.3f}")
print(f"📉 Uncertainty: mean={log_var.mean().item():.3f}")
print("="*100)

In [ ]:
# ## 6. Training Loop
# Встроенные веса классов + гетероскедастический лосс + калибровка

def train_exp1(model: nn.Module, train_loader: DataLoader, val_loader: DataLoader,
               config: Exp1Config, seed: int, class_weights: torch.Tensor, num_classes: int):
    model.to(device).float()
    optimizer = torch.optim.AdamW(model.parameters(), lr=config.lr, weight_decay=config.weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=config.epochs)
    scaler = GradScaler()
    best_f1, best_state, patience = 0.0, None, 0
    class_weights = class_weights.to(device)

    for epoch in range(config.epochs):
        model.train(); epoch_loss = 0.0
        for batch in tqdm(train_loader, desc=f"Seed {seed} | Epoch {epoch+1}", leave=False):
            batch = {k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in batch.items()}
            optimizer.zero_grad()
            with autocast():
                logits, log_var, _ = model(batch, training=True)
                ce = F.cross_entropy(logits, batch["label"], weight=class_weights)
                prec = torch.exp(-torch.clamp(log_var.squeeze(-1), min=-5, max=5))
                unc_loss = prec * ce + log_var.squeeze(-1)
                loss = ce + config.uncertainty_loss_weight * unc_loss.mean()
            scaler.scale(loss).backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer); scaler.update()
            epoch_loss += loss.item()

        model.eval(); preds, trues, probs, uncs, gates_a, gates_v = [], [], [], [], [], []
        with torch.no_grad():
            for batch in val_loader:
                batch = {k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in batch.items()}
                logits, log_var, gates = model(batch, training=False)
                probs_batch = torch.softmax(logits, dim=1)
                preds.extend(logits.argmax(1).cpu().numpy())
                trues.extend(batch["label"].cpu().numpy())
                probs.extend(probs_batch.cpu().numpy())
                uncs.extend(torch.exp(log_var.squeeze(-1)).cpu().numpy())
                gates_a.extend(gates.get("audio", torch.zeros(len(preds))).cpu().numpy())
                gates_v.extend(gates.get("video", torch.zeros(len(preds))).cpu().numpy())

        f1 = f1_score(trues, preds, average="macro", zero_division=0)
        scheduler.step()
        if f1 > best_f1:
            best_f1, patience = f1, 0
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
        else: patience += 1
        if patience >= config.early_stopping_patience: break

    if best_state: model.load_state_dict(best_state)
    torch.cuda.empty_cache(); gc.collect()
    return model, f1, np.array(preds), np.array(trues), np.array(probs), np.array(uncs), np.array(gates_a), np.array(gates_v)

In [ ]:
# ## 7. Visualization (Paper-Ready)
def plot_paper_metrics(val_probs, val_trues, val_unc, val_gates_a, val_gates_v, emotions, save_dir="./exp1_viz"):
    Path(save_dir).mkdir(exist_ok=True)
    preds = np.argmax(val_probs, axis=1)
    acc_mask = (preds == val_trues).astype(float)

    # 1. Reliability Diagram
    plt.figure(figsize=(5,4))
    conf = np.max(val_probs, axis=1)
    accs, confs = [], []
    for i in range(10):
        mask = (conf >= i/10) & (conf < (i+1)/10)
        if mask.sum() > 0:
            accs.append(acc_mask[mask].mean())
            confs.append((i+0.5)/10)
    plt.plot(confs, accs, 'o-', label='Model', linewidth=2)
    plt.plot([0,1], [0,1], 'k--', label='Perfect')
    plt.xlabel('Confidence'); plt.ylabel('Accuracy'); plt.title('Reliability Diagram'); plt.legend(); plt.grid()
    plt.savefig(f"{save_dir}/reliability.png", dpi=300)

    # 2. Gate Distribution
    fig, axes = plt.subplots(1, 2, figsize=(10,4))
    for i, emo in enumerate(emotions):
        mask = val_trues == i
        if mask.sum() < 5: continue
        sns.kdeplot(val_gates_a[mask], ax=axes[0], label=emo, fill=True, alpha=0.3)
        sns.kdeplot(val_gates_v[mask], ax=axes[1], label=emo, fill=True, alpha=0.3)
    axes[0].set_title('Audio Gate'); axes[1].set_title('Video Gate')
    for ax in axes: ax.set_xlabel('Gate Value'); ax.legend(fontsize=7)
    plt.tight_layout(); plt.savefig(f"{save_dir}/gates.png", dpi=300)

    # 3. Uncertainty vs Accuracy
    plt.figure(figsize=(5,4))
    plt.scatter(val_unc, acc_mask, alpha=0.3, s=8, edgecolors='k', linewidth=0.1)
    plt.xlabel('Uncertainty'); plt.ylabel('Correct (1/0)'); plt.grid()
    plt.savefig(f"{save_dir}/unc_vs_acc.png", dpi=300)

    ece = compute_ece(val_probs, val_trues)
    brier = compute_brier(val_probs, val_trues, len(emotions))
    macro_f1 = f1_score(val_trues, preds, average="macro", zero_division=0)
    print(f"📈 Final Metrics: F1={macro_f1:.3f} | ECE={ece:.3f} | Brier={brier:.3f}")
    plt.close('all')

In [ ]:
# ## 8. Run Experiment
def run_article_1():
    # Вычисляем веса один раз
    train_labels = np.array([d["label"].item() for d in train_ds])
    cw = compute_class_weight('balanced', classes=np.arange(len(train_ds.emotions)), y=train_labels)
    class_weights_tensor = torch.tensor(cw, dtype=torch.float32, device=device)

    for seed in range(42, 42 + config.n_seeds):
        set_seed(seed)
        model = UncertaintyGateModel(config, num_classes=len(train_ds.emotions))

        # Передаём class_weights_tensor
        _, f1, p, t, probs, unc, ga, gv = train_exp1(
            model, dl_train, dl_val, config, seed, class_weights_tensor, len(train_ds.emotions)
        )

        ece = compute_ece(probs, t)
        brier = compute_brier(probs, t, len(train_ds.emotions))
        all_f1.append(f1); all_ece.append(ece); all_brier.append(brier)
        last_seed_results = (probs, t, unc, ga, gv)
        print(f"  Seed {seed}: F1={f1:.3f} | ECE={ece:.3f}")

    metrics_df = pd.DataFrame({"Seed": range(42, 42+config.n_seeds), "F1": all_f1, "ECE": all_ece, "Brier": all_brier})
    print("\n📊 Per-Seed Results:")
    print(metrics_df.to_string(index=False))
    print(f"\n📈 Mean ± Std: F1={np.mean(all_f1):.3f}±{np.std(all_f1):.3f} | ECE={np.mean(all_ece):.3f}±{np.std(all_ece):.3f}")

    plot_paper_metrics(*last_seed_results, train_ds.emotions)
    metrics_df.to_csv(f"./{config.experiment_name}_results.csv", index=False)
    print("✅ Done. Results saved.")

if __name__ == "__main__":
    run_article_1()

In [ ]:
# ## 8. Run Experiment
def run_article_1():
    print("🚀 Starting Article #1: Uncertainty-Gated Fusion (REAL DATA)")

    # ✅ Вычисляем class_weights здесь (не в dataset)
    train_labels = np.array([d["label"].item() for d in train_ds])
    cw = compute_class_weight('balanced', classes=np.arange(len(train_ds.emotions)), y=train_labels)
    class_weights_tensor = torch.tensor(cw, dtype=torch.float32, device=device)

    all_f1, all_ece, all_brier = [], [], []
    last_seed_results = None

    for seed in range(42, 42 + config.n_seeds):
        set_seed(seed)
        model = UncertaintyGateModel(config, num_classes=len(train_ds.emotions))

        # ✅ Передаём class_weights_tensor и получаем 8 значений
        _, f1, p, t, probs, unc, ga, gv = train_exp1(
            model, dl_train, dl_val, config, seed, class_weights_tensor, len(train_ds.emotions)
        )

        ece = compute_ece(probs, t)
        brier = compute_brier(probs, t, len(train_ds.emotions))
        all_f1.append(f1); all_ece.append(ece); all_brier.append(brier)
        last_seed_results = (probs, t, unc, ga, gv)
        print(f"  Seed {seed}: F1={f1:.3f} | ECE={ece:.3f}")

    metrics_df = pd.DataFrame({"Seed": range(42, 42+config.n_seeds), "F1": all_f1, "ECE": all_ece, "Brier": all_brier})
    print("\n📊 Per-Seed Results:")
    print(metrics_df.to_string(index=False))
    print(f"\n📈 Mean ± Std: F1={np.mean(all_f1):.3f}±{np.std(all_f1):.3f} | ECE={np.mean(all_ece):.3f}±{np.std(all_ece):.3f}")

    plot_paper_metrics(*last_seed_results, train_ds.emotions)
    metrics_df.to_csv(f"./{config.experiment_name}_results.csv", index=False)
    print("✅ Done. Results saved.")

if __name__ == "__main__":
    run_article_1()

🚀 Starting Article #1: Uncertainty-Gated Fusion (REAL DATA)


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Seed 42: F1=0.122 | ECE=0.016


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Seed 43: F1=0.097 | ECE=0.072


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Seed 44: F1=0.129 | ECE=0.027

📊 Per-Seed Results:
 Seed       F1      ECE    Brier
   42 0.121735 0.016480 0.827901
   43 0.097238 0.071597 0.847746
   44 0.128944 0.027366 0.831476

📈 Mean ± Std: F1=0.116±0.014 | ECE=0.038±0.024
📈 Final Metrics: F1=0.129 | ECE=0.027 | Brier=0.831
✅ Done. Results saved.
